# Block 3. MODIS against VIIRS in the overlap

Goal: choose the method that turns MODIS counts into VIIRS-equivalent counts.
Plan: `docs/eda-plan.md`, block 3. Results: `docs/findings/03-modis-vs-viirs.md`.

Everything here uses the rows kept by the quality filter of block 1 (`quality_flags.parquet`),
`burn_date` as the day key, and Suomi-NPP as the VIIRS reference.

In [1]:
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning)  # duckdb -> pandas noise
pd.set_option("display.width", 200, "display.max_columns", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "NASA_Data_Sources").is_dir())
con = duckdb.connect()
con.execute("SET TimeZone='UTC'")
DATA = ROOT / "NASA_Data_Sources" / "processed"
KEY = "satellite, acq_datetime, latitude, longitude"
# Only rows that pass the quality filter of block 1 (docs/findings/01-quality.md).
con.execute(f"""
    CREATE TABLE h AS
    SELECT t.burn_date, t.burn_year, t.burn_month, t.pass_slot, t.latitude, t.longitude, t.country,
           coalesce(q.admin1_fixed, t.admin1) AS admin1, t.instrument, t.satellite
    FROM read_parquet('{DATA}/hotspots/*/*.parquet', hive_partitioning=true) t
    JOIN read_parquet('{DATA}/quality/quality_flags.parquet') q USING ({KEY})
    WHERE q.keep
""")



def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

## 3.1 Ratio over time

In [2]:
yearly = q("""
SELECT burn_year AS year,
       count(*) FILTER (WHERE instrument = 'MODIS') AS modis,
       count(*) FILTER (WHERE satellite = 'Aqua') AS aqua,
       count(*) FILTER (WHERE satellite = 'Terra') AS terra,
       count(*) FILTER (WHERE satellite = 'Suomi-NPP') AS snpp,
       count(*) FILTER (WHERE satellite = 'NOAA-20') AS noaa20
FROM h WHERE burn_year BETWEEN 2013 AND 2025 GROUP BY 1 ORDER BY 1
""").set_index("year")
ratios = pd.DataFrame({
    "snpp / modis": yearly.snpp / yearly.modis,
    "snpp / aqua": yearly.snpp / yearly.aqua,
    "aqua / terra": yearly.aqua / yearly.terra,
    "noaa20 / snpp": (yearly.noaa20 / yearly.snpp).where(yearly.index >= 2019),
}).round(2)
ratios.T

year,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
snpp / modis,3.31,3.11,3.21,3.09,3.11,3.10,2.93,3.40,3.12,3.69,3.19,2.82,3.28
snpp / aqua,5.17,5.07,4.92,4.72,4.95,4.80,4.53,5.22,4.88,5.74,5.25,4.17,4.71
aqua / terra,1.78,1.59,1.87,1.89,1.69,1.82,1.83,1.87,1.77,1.81,1.55,2.08,2.31
noaa20 / snpp,NaN,NaN,NaN,NaN,NaN,NaN,0.99,0.97,1.01,0.98,0.95,1.03,0.99


## 3.2 Like for like: Aqua against Suomi-NPP per pass, and what the ratio depends on (3.3)

In [3]:
q("""
SELECT country, pass_slot,
       count(*) FILTER (WHERE satellite = 'Suomi-NPP') AS snpp,
       count(*) FILTER (WHERE satellite = 'Aqua') AS aqua,
       round(count(*) FILTER (WHERE satellite = 'Suomi-NPP') / count(*) FILTER (WHERE satellite = 'Aqua'), 2) AS ratio
FROM h WHERE burn_year BETWEEN 2013 AND 2021 GROUP BY ALL ORDER BY ALL
""")

,country,pass_slot,snpp,aqua,ratio
0,Argentina,day_pass,1230784,310073,3.97
1,Argentina,night_pass,431873,22757,18.98
2,Chile,day_pass,124469,36291,3.43
3,Chile,night_pass,65697,4924,13.34


In [4]:
by_region = q("""
SELECT country, admin1,
       count(*) FILTER (WHERE satellite = 'Suomi-NPP') AS snpp,
       count(*) FILTER (WHERE instrument = 'MODIS') AS modis
FROM h WHERE burn_year BETWEEN 2013 AND 2021 AND admin1 IS NOT NULL GROUP BY ALL
""")
by_region["ratio"] = (by_region.snpp / by_region.modis).round(2)
by_region = by_region[by_region.modis >= 500].sort_values("ratio")
print("Regions with 500+ MODIS hotspots:", len(by_region),
      "| ratio from", by_region.ratio.min(), "to", by_region.ratio.max())
pd.concat([by_region.head(6), by_region.tail(6)])

Regions with 500+ MODIS hotspots: 33 | ratio from 2.08 to 5.88


,country,admin1,snpp,modis,ratio
45,Chile,Región de Los Ríos,2614,1256,2.08
43,Argentina,Buenos Aires,54722,24309,2.25
28,Chile,Región de Ñuble,21578,8139,2.65
41,Chile,Región del Bío-Bío,31726,11745,2.70
42,Chile,Región de La Araucanía,45357,16358,2.77
7,Argentina,Tucumán,40768,14684,2.78
32,Argentina,Catamarca,12465,3233,3.86
21,Chile,Región de Aysén del Gral.Ibañez del Campo,2172,558,3.89
29,Chile,Región Metropolitana de Santiago,8168,1983,4.12
35,Argentina,Chubut,11813,2817,4.19


The ratio also depends on how much fire there is in the month: big months have a higher ratio.

In [5]:
monthly = q("""
SELECT country, coalesce(admin1, '(no region)') AS admin1, burn_year AS year, burn_month AS month,
       count(*) FILTER (WHERE satellite = 'Suomi-NPP') AS v,
       count(*) FILTER (WHERE instrument = 'MODIS') AS m
FROM h WHERE burn_date >= DATE '2012-02-01' AND burn_year <= 2025 GROUP BY ALL
""")
size = monthly[(monthly.m >= 20) & monthly.year.le(2021)].copy()
size["modis_bin"] = pd.cut(size.m, [20, 50, 200, 1000, 5000, 1e9], right=False,
                           labels=["20-49", "50-199", "200-999", "1000-4999", "5000+"])
size.groupby(["country", "modis_bin"], observed=True).apply(
    lambda d: pd.Series({"region_months": len(d), "ratio": round(d.v.sum() / d.m.sum(), 2)}),
    include_groups=False,
).unstack(0)

region_months            ratio      
country       Argentina  Chile Argentina Chile
modis_bin                                     
20-49             437.0  223.0      3.29  2.89
50-199            781.0  236.0      3.06  2.67
200-999           468.0   60.0      3.13  3.07
1000-4999         145.0    3.0      3.23  3.22
5000+               2.0    1.0      3.44  3.21

## 3.9 NOAA-20 against Suomi-NPP

NOAA-20 replaces Suomi-NPP from November 2026. Is a factor needed?

In [6]:
q("""
SELECT country, burn_year AS year,
       round(count(*) FILTER (WHERE satellite = 'NOAA-20') / count(*) FILTER (WHERE satellite = 'Suomi-NPP'), 3) AS noaa20_over_snpp
FROM h WHERE burn_year BETWEEN 2019 AND 2025 GROUP BY ALL ORDER BY ALL
""").pivot(index="year", columns="country", values="noaa20_over_snpp")

country,Argentina,Chile
year,,
2019,0.993,0.987
2020,0.975,0.964
2021,1.005,1.009
2022,0.986,0.946
2023,0.950,0.930
2024,1.030,1.006
2025,0.993,0.969


## 3.10 Harmonization candidates, scored with leave-one-year-out

Each candidate predicts the VIIRS count of a region and month from its MODIS count.

| Candidate | Factor |
|---|---|
| `country_month` | one per country and calendar month (v0) |
| `region` | one per region, shrunk towards the country factor |
| `region_month` | one per region and calendar month, shrunk towards the country-month factor |
| `power` | VIIRS = a(region) × MODIS^b(country): the ratio is allowed to grow with the size of the month |

Shrinkage: `(V + k × prior) / (M + k)`, so a region with few detections stays close to the prior.
Score: WAPE = sum of absolute errors / sum of VIIRS, over region-months. It weighs big months more.

In [7]:
K = 200  # pseudo-count of MODIS detections behind the prior


def fit_country_month(tr):
    f = tr.groupby(["country", "month"])[["v", "m"]].sum()
    f = (f.v / f.m).rename("f").reset_index()
    return lambda d: d.merge(f, how="left", on=["country", "month"]).eval("m * f").to_numpy()


def fit_region(tr, k=K):
    c = tr.groupby("country")[["v", "m"]].sum()
    r = tr.groupby(["country", "admin1"])[["v", "m"]].sum().reset_index()
    r["f"] = (r.v + k * r.country.map(c.v / c.m)) / (r.m + k)
    cf = (c.v / c.m).rename("cf").reset_index()

    def predict(d):
        x = d.merge(r[["country", "admin1", "f"]], how="left", on=["country", "admin1"]).merge(cf, on="country")
        return (x.m * x.f.fillna(x.cf)).to_numpy()
    return predict


def fit_region_month(tr, k=K):
    cm = tr.groupby(["country", "month"])[["v", "m"]].sum()
    cm = (cm.v / cm.m).rename("cf").reset_index()
    rm = tr.groupby(["country", "admin1", "month"])[["v", "m"]].sum().reset_index().merge(cm, on=["country", "month"])
    rm["f"] = (rm.v + k * rm.cf) / (rm.m + k)

    def predict(d):
        x = d.merge(rm[["country", "admin1", "month", "f"]], how="left", on=["country", "admin1", "month"])
        x = x.merge(cm, how="left", on=["country", "month"])
        return (x.m * x.f.fillna(x.cf)).to_numpy()
    return predict


def fit_power(tr, min_obs=12):
    params = {}
    for country, d in tr[(tr.m >= 10) & (tr.v > 0)].groupby("country"):
        x, y = np.log(d.m), np.log(d.v)
        xd, yd = x - x.groupby(d.admin1).transform("mean"), y - y.groupby(d.admin1).transform("mean")
        b = float((xd * yd).sum() / (xd**2).sum())
        resid = y - b * x
        a = resid.groupby(d.admin1).mean()[resid.groupby(d.admin1).size() >= min_obs]
        params[country] = (b, a, float(resid.mean()))
    scale = {}

    def raw(d):
        out = np.zeros(len(d))
        for country, (b, a, a0) in params.items():
            rows = (d.country == country).to_numpy()
            ar = d.admin1[rows].map(a).fillna(a0).to_numpy()
            out[rows] = np.exp(ar) * d.m[rows].to_numpy() ** b
        return np.where(d.m.to_numpy() > 0, out, 0.0)

    pred_tr = pd.Series(raw(tr), index=tr.index)
    for country in params:  # undo the bias of fitting in logs: totals must match on the fit years
        rows = tr.country == country
        scale[country] = tr.v[rows].sum() / pred_tr[rows].sum()
    return lambda d: raw(d) * d.country.map(scale).to_numpy()


CANDIDATES = {"country_month": fit_country_month, "region": fit_region,
              "region_month": fit_region_month, "power": fit_power}
fit_years = monthly[monthly.year <= 2021].reset_index(drop=True)
test_years = monthly[monthly.year >= 2022].reset_index(drop=True)

loyo = fit_years.copy()
for name, fit in CANDIDATES.items():
    loyo[name] = np.nan
    for year in sorted(loyo.year.unique()):
        held = loyo.year == year
        loyo.loc[held, name] = fit(loyo[~held].reset_index(drop=True))(loyo[held].reset_index(drop=True))
test = test_years.copy()
for name, fit in CANDIDATES.items():
    test[name] = fit(fit_years)(test)


def score(d, label):
    rows = []
    for country, g in d.groupby("country"):
        for name in CANDIDATES:
            year_err = g.groupby("year")[[name, "v"]].sum()
            rows.append({
                "set": label, "country": country, "candidate": name,
                "wape_region_month_pct": 100 * (g[name] - g.v).abs().sum() / g.v.sum(),
                "mean_abs_yearly_error_pct": 100 * (year_err[name] / year_err.v - 1).abs().mean(),
                "worst_year_error_pct": 100 * (year_err[name] / year_err.v - 1).abs().max(),
            })
    return pd.DataFrame(rows)


scores = pd.concat([score(loyo, "leave-one-year-out 2012-2021"), score(test, "test 2022-2025")])
scores.round(1).set_index(["set", "country", "candidate"])

wape_region_month_pct  mean_abs_yearly_error_pct  worst_year_error_pct
set                          country   candidate                                                                            
leave-one-year-out 2012-2021 Argentina country_month                   16.7                        3.8                   8.9
                                       region                          15.9                        3.9                   8.7
                                       region_month                    15.1                        3.2                   7.9
                                       power                           16.8                        3.7                   9.4
                             Chile     country_month                   25.3                        5.0                   9.6
                                       region                          27.6                        5.0                  16.4
                                       region_month                    24.6                        4.8                  14.7
                                       power                           32.4                        6.5                  20.2
test 2022-2025               Argentina country_month                   23.4                        7.2                  14.0
                                       region                          22.4                        7.1                  13.9
                                       region_month                    21.3                        7.2                  14.2
                                       power                           23.5                        7.4                  14.2
                             Chile     country_month                   27.8                        8.8                  14.5
                                       region                          31.8                       10.4                  14.0
                                       region_month                    24.6                        8.4                  17.7
                                       power                           32.0                       10.7                  17.6

## 3.11 Is the ratio stable enough to extrapolate back to 2000-2011?

Fit on the first half of the stable overlap (2012-2016), predict the second half (2017-2021), and the reverse.

In [8]:
halves = []
for name, fit in CANDIDATES.items():
    for tr_years, te_years in [((2012, 2016), (2017, 2021)), ((2017, 2021), (2012, 2016))]:
        tr = fit_years[fit_years.year.between(*tr_years)].reset_index(drop=True)
        te = fit_years[fit_years.year.between(*te_years)].reset_index(drop=True)
        pred = fit(tr)(te)
        halves.append({"candidate": name, "fit": f"{tr_years[0]}-{tr_years[1]}", "predict": f"{te_years[0]}-{te_years[1]}",
                       "total_error_pct": 100 * (pred.sum() / te.v.sum() - 1),
                       "wape_region_month_pct": 100 * np.abs(pred - te.v).sum() / te.v.sum()})
pd.DataFrame(halves).round(1)

,candidate,fit,predict,total_error_pct,wape_region_month_pct
0,country_month,2012-2016,2017-2021,-0.4,17.6
1,country_month,2017-2021,2012-2016,0.2,17.1
2,region,2012-2016,2017-2021,-0.6,17.2
3,region,2017-2021,2012-2016,-1.7,17.7
4,region_month,2012-2016,2017-2021,-1.0,16.2
5,region_month,2017-2021,2012-2016,-1.6,17.3
6,power,2012-2016,2017-2021,1.0,19.2
7,power,2017-2021,2012-2016,-2.5,18.1


## Fire-days: a second metric that depends less on pixel size (3.8)

In [9]:
q("""
SELECT burn_year AS year,
       count(DISTINCT (floor(latitude * 4), floor(longitude * 4), burn_date)) FILTER (WHERE satellite = 'Suomi-NPP') AS snpp_fire_days,
       count(DISTINCT (floor(latitude * 4), floor(longitude * 4), burn_date)) FILTER (WHERE instrument = 'MODIS') AS modis_fire_days
FROM h WHERE burn_year BETWEEN 2013 AND 2025 GROUP BY 1 ORDER BY 1
""").assign(ratio=lambda d: (d.snpp_fire_days / d.modis_fire_days).round(2)).set_index("year").T

year,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
snpp_fire_days,53169.00,34060.00,42125.00,42946.00,42883.00,44948.00,42546.00,74082.0,48592.00,51795.00,40285.00,38348.00,42158.00
modis_fire_days,26667.00,16450.00,21140.00,22726.00,23664.00,23457.00,22776.00,41168.0,24898.00,26729.00,20961.00,22263.00,21539.00
ratio,1.99,2.07,1.99,1.89,1.81,1.92,1.87,1.8,1.95,1.94,1.92,1.72,1.96


## 3.10b Does it help to treat the passes separately?

MODIS misses most night fires (3.2), so the night share of a month could explain part of the error.
Same method as `region_month`, with a different MODIS input:

| Variant | Predicts the VIIRS count of a region and month from |
|---|---|
| `all MODIS` | Terra + Aqua, day and night (the chosen method) |
| `MODIS day pass` | Terra + Aqua, day pass only |
| `Aqua only` | Aqua, day and night (same orbit as Suomi-NPP) |
| `per pass` | day and night predicted separately, each from its own MODIS count, then added |

In [10]:
by_pass = q("""
SELECT country, coalesce(admin1, '(no region)') AS admin1, burn_year AS year, burn_month AS month,
       pass_slot::VARCHAR AS pass_slot,
       count(*) FILTER (WHERE satellite = 'Suomi-NPP') AS v,
       count(*) FILTER (WHERE instrument = 'MODIS') AS m,
       count(*) FILTER (WHERE satellite = 'Aqua') AS aqua
FROM h WHERE burn_date >= DATE '2012-02-01' AND burn_year <= 2025 GROUP BY ALL
""")
KEYS = ["country", "admin1", "year", "month"]
total = by_pass.groupby(KEYS, as_index=False)[["v", "m", "aqua"]].sum()
day = by_pass[by_pass.pass_slot == "day_pass"].groupby(KEYS, as_index=False).m.sum().rename(columns={"m": "m_day"})
total = total.merge(day, how="left", on=KEYS).fillna({"m_day": 0})


def predict(train, held, variant):
    """VIIRS prediction per region-month of `held`, as a frame with KEYS + pred."""
    if variant == "per pass":
        parts = []
        for slot in ["day_pass", "night_pass"]:
            tr = train[train.pass_slot == slot].reset_index(drop=True)
            te = held[held.pass_slot == slot].reset_index(drop=True)
            parts.append(te.assign(pred=fit_region_month(tr)(te)))
        return pd.concat(parts).groupby(KEYS, as_index=False).pred.sum()
    column = {"all MODIS": "m", "MODIS day pass": "m_day", "Aqua only": "aqua"}[variant]
    tr = train.assign(m=train[column]).reset_index(drop=True)
    te = held.assign(m=held[column]).reset_index(drop=True)
    return te[KEYS].assign(pred=fit_region_month(tr)(te))


VARIANTS = ["all MODIS", "MODIS day pass", "Aqua only", "per pass"]
rows = []
for variant in VARIANTS:
    data = by_pass if variant == "per pass" else total
    fit_part, test_part = data[data.year <= 2021], data[data.year >= 2022]
    folds = [predict(fit_part[fit_part.year != y], fit_part[fit_part.year == y], variant)
             for y in sorted(fit_part.year.unique())]
    for label, pred in [("leave-one-year-out 2012-2021", pd.concat(folds)),
                        ("test 2022-2025", predict(fit_part, test_part, variant))]:
        d = total[KEYS + ["v"]].merge(pred, on=KEYS)
        for country, g in d.groupby("country"):
            yearly_err = g.groupby("year")[["pred", "v"]].sum()
            rows.append({"set": label, "country": country, "variant": variant,
                         "wape_region_month_pct": 100 * (g.pred - g.v).abs().sum() / g.v.sum(),
                         "mean_abs_yearly_error_pct": 100 * (yearly_err.pred / yearly_err.v - 1).abs().mean()})
pd.DataFrame(rows).round(1).set_index(["set", "country", "variant"]).unstack("variant")

wape_region_month_pct                                   mean_abs_yearly_error_pct                                  
variant                                            Aqua only MODIS day pass all MODIS per pass                 Aqua only MODIS day pass all MODIS per pass
set                          country                                                                                                                      
leave-one-year-out 2012-2021 Argentina                  16.2           18.0      15.1     13.9                       3.4            4.3       3.2      2.1
                             Chile                      26.7           30.5      24.6     24.0                       4.9            6.4       4.8      5.3
test 2022-2025               Argentina                  23.3           25.5      21.3     20.0                       9.5            9.1       7.2      7.7
                             Chile                      26.8           35.7      24.6     27.7                      11.5           12.6       8.4      9.7